# EOQ Event-Driven Inventory Simulation

This notebook studies a single-stock inventory system with:

- EOQ for order quantity selection
- continuous review using a reorder point
- event-driven demand and replenishment events
- interactive controls built with `ipywidgets`
- one-factor-at-a-time sensitivity analysis


## Mathematical model

For annual demand $D$, ordering cost $K$, and annual holding cost $h$, the classical EOQ is

$$
Q^* = \sqrt{\frac{2DK}{h}}.
$$

With lead time $L$ days and daily demand $d = D/365$, a natural deterministic reorder point is

$$
R = dL.
$$

The simulation below uses `simpy`. The clock jumps only to demand events and replenishment arrivals.


In [13]:
from pathlib import Path
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from eoq_simulation import EOQParameters, run_event_driven_simulation


In [14]:
def build_params(
    annual_demand=1200,
    ordering_cost=80.0,
    holding_cost=4.0,
    lead_time_days=7.0,
    use_manual_reorder_point=False,
    manual_reorder_point=25,
    horizon_days=180,
    initial_inventory=60,
    use_manual_order_quantity=False,
    manual_order_quantity=120,
):
    return EOQParameters(
        annual_demand=float(annual_demand),
        ordering_cost=float(ordering_cost),
        holding_cost=float(holding_cost),
        lead_time_days=float(lead_time_days),
        reorder_point=float(manual_reorder_point) if use_manual_reorder_point else None,
        horizon_days=float(horizon_days),
        initial_inventory=float(initial_inventory),
        order_quantity=float(manual_order_quantity) if use_manual_order_quantity else None,
    )


def simulate_and_plot(
    annual_demand=1200,
    ordering_cost=80.0,
    holding_cost=4.0,
    lead_time_days=7.0,
    use_manual_reorder_point=False,
    manual_reorder_point=25,
    horizon_days=180,
    initial_inventory=60,
    use_manual_order_quantity=False,
    manual_order_quantity=120,
):
    params = build_params(
        annual_demand=annual_demand,
        ordering_cost=ordering_cost,
        holding_cost=holding_cost,
        lead_time_days=lead_time_days,
        use_manual_reorder_point=use_manual_reorder_point,
        manual_reorder_point=manual_reorder_point,
        horizon_days=horizon_days,
        initial_inventory=initial_inventory,
        use_manual_order_quantity=use_manual_order_quantity,
        manual_order_quantity=manual_order_quantity,
    )

    result = run_event_driven_simulation(params)
    timeline = pd.DataFrame(result['timeline'])
    orders = pd.DataFrame(result['orders'])
    summary = pd.Series(result['summary']).rename('value').to_frame()

    

    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].step(timeline['time_days'], timeline['inventory_on_hand'], where='post', linewidth=2)
    axes[0].axhline(params.reorder_level, color='tab:red', linestyle='--', label='reorder point')
    axes[0].set_ylabel('Inventory on hand')
    axes[0].set_title('Inventory trajectory')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    served = timeline[timeline['event'] == 'demand_served']
    lost = timeline[timeline['event'] == 'demand_lost']
    repl = timeline[timeline['event'] == 'replenishment_arrived']
    axes[1].scatter(served['time_days'], [1] * len(served), label='served demand', marker='o', s=16)
    axes[1].scatter(lost['time_days'], [0.5] * len(lost), label='lost demand', marker='x', s=28)
    axes[1].scatter(repl['time_days'], [1.5] * len(repl), label='replenishment', marker='s', s=28)
    axes[1].set_yticks([0.5, 1.0, 1.5], ['lost', 'served', 'arrival'])
    axes[1].set_xlabel('Time (days)')
    axes[1].set_title('Event timeline')
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    if not orders.empty:
        display(orders.style.format('{:,.2f}'))
    display(summary.style.format('{:,.3f}'))
    display(
        timeline.tail(20).style.format(
            '{:,.2f}',
            subset=['time_days', 'inventory_on_hand', 'inventory_position', 'orders_outstanding'],
        )
    )


In [11]:
controls = widgets.interactive(
    simulate_and_plot,
    annual_demand=widgets.IntSlider(value=1200, min=120, max=10000, step=10, description='D/year'),
    ordering_cost=widgets.FloatSlider(value=80.0, min=5.0, max=300.0, step=5.0, description='K'),
    holding_cost=widgets.FloatSlider(value=4.0, min=0.5, max=30.0, step=0.5, description='h'),
    lead_time_days=widgets.FloatSlider(value=7.0, min=0.0, max=45.0, step=1.0, description='Lead'),
    use_manual_reorder_point=widgets.Checkbox(value=False, description='Manual R'),
    manual_reorder_point=widgets.IntSlider(value=25, min=0, max=300, step=1, description='R'),
    horizon_days=widgets.IntSlider(value=180, min=30, max=730, step=10, description='Horizon'),
    initial_inventory=widgets.IntSlider(value=60, min=0, max=500, step=1, description='Initial'),
    use_manual_order_quantity=widgets.Checkbox(value=False, description='Manual Q'),
    manual_order_quantity=widgets.IntSlider(value=120, min=1, max=1000, step=1, description='Q'),
)

display(controls)


interactive(children=(IntSlider(value=1200, description='D/year', max=10000, min=120, step=10), FloatSlider(va…

## Sensitivity analysis

This section performs a one-factor-at-a-time study. Choose one input, define a range, and the notebook will rerun the EOQ simulation for each value while holding the other inputs fixed.


In [4]:
def run_sensitivity_analysis(
    parameter_name='annual_demand',
    start_value=600.0,
    end_value=1800.0,
    num_points=7,
    annual_demand=1200,
    ordering_cost=80.0,
    holding_cost=4.0,
    lead_time_days=7.0,
    use_manual_reorder_point=False,
    manual_reorder_point=25,
    horizon_days=180,
    initial_inventory=60,
    use_manual_order_quantity=False,
    manual_order_quantity=120,
):
    base_inputs = {
        'annual_demand': float(annual_demand),
        'ordering_cost': float(ordering_cost),
        'holding_cost': float(holding_cost),
        'lead_time_days': float(lead_time_days),
        'use_manual_reorder_point': bool(use_manual_reorder_point),
        'manual_reorder_point': float(manual_reorder_point),
        'horizon_days': float(horizon_days),
        'initial_inventory': float(initial_inventory),
        'use_manual_order_quantity': bool(use_manual_order_quantity),
        'manual_order_quantity': float(manual_order_quantity),
    }

    if num_points < 2:
        raise ValueError('num_points must be at least 2')

    sweep_values = pd.Series(
        [start_value + i * (end_value - start_value) / (num_points - 1) for i in range(num_points)],
        name=parameter_name,
    )

    records = []
    for value in sweep_values:
        current_inputs = dict(base_inputs)
        current_inputs[parameter_name] = float(value)
        params = build_params(**current_inputs)
        result = run_event_driven_simulation(params)
        record = dict(result['summary'])
        record[parameter_name] = value
        records.append(record)

    results_df = pd.DataFrame(records)
    display(results_df.style.format('{:,.3f}'))

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    plot_metrics = [
        ('annual_total_cost', 'Annual total cost'),
        ('fill_rate', 'Fill rate'),
        ('average_inventory', 'Average inventory'),
        ('orders_placed', 'Orders placed'),
    ]

    for axis, (metric, title) in zip(axes.flat, plot_metrics):
        axis.plot(results_df[parameter_name], results_df[metric], marker='o', linewidth=2)
        axis.set_title(title)
        axis.set_xlabel(parameter_name)
        axis.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    return results_df


In [12]:
sensitivity_controls = widgets.interactive(
    run_sensitivity_analysis,
    parameter_name=widgets.Dropdown(
        options=['annual_demand', 'ordering_cost', 'holding_cost', 'lead_time_days', 'manual_reorder_point', 'manual_order_quantity'],
        value='annual_demand',
        description='Sweep',
    ),
    start_value=widgets.FloatText(value=600.0, description='Start'),
    end_value=widgets.FloatText(value=1800.0, description='End'),
    num_points=widgets.IntSlider(value=7, min=2, max=25, step=1, description='Points'),
    annual_demand=widgets.IntSlider(value=1200, min=120, max=10000, step=10, description='D/year'),
    ordering_cost=widgets.FloatSlider(value=80.0, min=5.0, max=300.0, step=5.0, description='K'),
    holding_cost=widgets.FloatSlider(value=4.0, min=0.5, max=30.0, step=0.5, description='h'),
    lead_time_days=widgets.FloatSlider(value=7.0, min=0.0, max=45.0, step=1.0, description='Lead'),
    use_manual_reorder_point=widgets.Checkbox(value=False, description='Manual R'),
    manual_reorder_point=widgets.FloatSlider(value=25.0, min=0.0, max=300.0, step=1.0, description='R'),
    horizon_days=widgets.IntSlider(value=180, min=30, max=730, step=10, description='Horizon'),
    initial_inventory=widgets.IntSlider(value=60, min=0, max=500, step=1, description='Initial'),
    use_manual_order_quantity=widgets.Checkbox(value=False, description='Manual Q'),
    manual_order_quantity=widgets.FloatSlider(value=120.0, min=1.0, max=1000.0, step=1.0, description='Q'),
)

display(sensitivity_controls)


interactive(children=(Dropdown(description='Sweep', options=('annual_demand', 'ordering_cost', 'holding_cost',…